In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
spark.sql("CREATE DATABASE IF NOT EXISTS practice_db")
spark.sql("USE practice_db")


In [0]:
# ── ORDERS TABLE (1M rows) ──────────────────────────────────────
orders_df = spark.range(1, 1_000_001).select(
    col("id").alias("order_id"),
    (col("id") % 100_000 + 1).alias("customer_id"),
    (col("id") % 500 + 1).alias("product_id"),
    (col("id") % 50 + 1).alias("store_id"),
    expr("date_add('2022-01-01', cast(id % 730 as int))").alias("order_date"),
    (rand() * 4900 + 100).alias("amount"),
    when(col("id") % 5 == 0, "CANCELLED")
     .when(col("id") % 3 == 0, "RETURNED")
     .otherwise("COMPLETED").alias("status"),
    (col("id") % 5 + 1).alias("rating"),
    lit("USD").alias("currency")
)
orders_df.write.mode("overwrite").saveAsTable("orders")

In [0]:
%sql
select * from practice_db.orders limit 50 ;

In [0]:

# ── CUSTOMERS TABLE (100K rows) ─────────────────────────────────
customers_df = spark.range(1, 100_001).select(
    col("id").alias("customer_id"),
    concat(lit("Customer_"), col("id")).alias("name"),
    expr("CASE WHEN id % 4 = 0 THEN 'Gold' WHEN id % 4 = 1 THEN 'Silver' WHEN id % 4 = 2 THEN 'Bronze' ELSE 'Standard' END").alias("tier"),
    expr("date_add('1970-01-01', cast(id % 18250 + 6570 as int))").alias("dob"),
    (col("id") % 30 + 1).alias("city_id"),
    expr("CASE WHEN id % 2 = 0 THEN 'M' ELSE 'F' END").alias("gender"),
    expr("date_add('2018-01-01', cast(id % 1825 as int))").alias("registered_date")
)
customers_df.write.mode("overwrite").saveAsTable("customers")


In [0]:
%sql
select * from practice_db.customers limit 10;

In [0]:

# ── PRODUCTS TABLE (500 rows) ───────────────────────────────────
products_df = spark.range(1, 501).select(
    col("id").alias("product_id"),
    concat(lit("Product_"), col("id")).alias("product_name"),
    expr("CASE WHEN id % 5 = 0 THEN 'Electronics' WHEN id % 5 = 1 THEN 'Clothing' WHEN id % 5 = 2 THEN 'Food' WHEN id % 5 = 3 THEN 'Books' ELSE 'Sports' END").alias("category"),
    (rand() * 990 + 10).alias("price"),
    (col("id") % 50 + 1).alias("supplier_id"),
    expr("CASE WHEN id % 3 = 0 THEN 'Discontinued' ELSE 'Active' END").alias("status")
)
products_df.write.mode("overwrite").saveAsTable("products")


In [0]:
%sql
select * from practice_db.products limit 10;

In [0]:

# ── EVENTS / CLICKSTREAM TABLE (5M rows — for streaming sim) ───
events_df = spark.range(1, 5_000_001).select(
    col("id").alias("event_id"),
    (col("id") % 100_000 + 1).alias("user_id"),
    expr("CASE WHEN id % 6 = 0 THEN 'purchase' WHEN id % 6 = 1 THEN 'view' WHEN id % 6 = 2 THEN 'cart_add' WHEN id % 6 = 3 THEN 'search' WHEN id % 6 = 4 THEN 'click' ELSE 'page_view' END").alias("event_type"),
    (col("id") % 500 + 1).alias("item_id"),
    expr("to_timestamp(date_add('2023-01-01', cast(id % 365 as int)))").alias("event_ts"),
    expr("concat('session_', cast(id % 500000 as string))").alias("session_id"),
    expr("CASE WHEN id % 3 = 0 THEN 'mobile' WHEN id % 3 = 1 THEN 'desktop' ELSE 'tablet' END").alias("device")
)
events_df.write.mode("overwrite").partitionBy("event_type").saveAsTable("events")


In [0]:
%sql
select * from practice_db.events limit 10

In [0]:

# '''# ── SKEWED TABLE (for skew practice) ───────────────────────────
# skewed_df = spark.range(1, 2_000_001).select(
#     col("id"),
#     expr("CASE WHEN id % 1000 < 5 THEN 1 ELSE id % 1000 END").alias("skew_key"),
#     (rand() * 100).alias("value")
# )
# skewed_df.write.mode("overwrite").saveAsTable("skewed_data")'''

print("All tables created. Verify:")
spark.sql("SHOW TABLES IN practice_db").show()

In [0]:
# %sql
# SELECT 'orders' as tbl, count(*) as rows FROM practice_db.orders
# UNION ALL
# SELECT 'customers', count(*) FROM practice_db.customers
# UNION ALL
# SELECT 'products', count(*) FROM practice_db.products
# UNION ALL
# SELECT 'events', count(*) FROM practice_db.events
# UNION ALL
# SELECT 'skewed_data', count(*) FROM practice_db.skewed_data;

In [0]:
# %python
# # Convert to Delta and create multiple versions
# spark.sql("CONVERT TO DELTA practice_db.orders")

# # Simulate version history
# spark.sql("""
#   UPDATE practice_db.orders 
#   SET status = 'PROCESSING' 
#   WHERE order_id BETWEEN 1 AND 10000
# """)
# spark.sql("""
#   DELETE FROM practice_db.orders 
#   WHERE status = 'CANCELLED' AND order_id < 5000
# """)

# # Now you have 3+ versions for time travel
# spark.sql("DESCRIBE HISTORY practice_db.orders").show(5, truncate=False)